In [1]:
import pandas as pd

print("Pandas version:", pd.__version__)

Pandas version: 3.0.6


In [3]:
file_path = "data/yellow_tripdata_2021-01.csv.gz"

df = pd.read_csv(file_path, nrows=100)

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,1,N,142,43,2,8.0,3.0,0.5,0.00,0.0,0.3,11.80,2.5
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,1,N,238,151,2,3.0,0.5,0.5,0.00,0.0,0.3,4.30,0.0
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,1,N,132,165,1,42.0,0.5,0.5,8.65,0.0,0.3,51.95,0.0
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0,10.60,1,N,138,132,1,29.0,0.5,0.5,6.05,0.0,0.3,36.35,0.0
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,1,N,68,33,1,16.5,0.5,0.5,4.06,0.0,0.3,24.36,2.5


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               100 non-null    int64  
 1   tpep_pickup_datetime   100 non-null    str    
 2   tpep_dropoff_datetime  100 non-null    str    
 3   passenger_count        100 non-null    int64  
 4   trip_distance          100 non-null    float64
 5   RatecodeID             100 non-null    int64  
 6   store_and_fwd_flag     100 non-null    str    
 7   PULocationID           100 non-null    int64  
 8   DOLocationID           100 non-null    int64  
 9   payment_type           100 non-null    int64  
 10  fare_amount            100 non-null    float64
 11  extra                  100 non-null    float64
 12  mta_tax                100 non-null    float64
 13  tip_amount             100 non-null    float64
 14  tolls_amount           100 non-null    float64
 15  improvement_surcha

In [6]:
chunks = pd.read_csv(file_path, chunksize=100_000)

for chunk in chunks:
    print(len(chunk))

100000
100000
100000
100000
100000
100000
100000
100000
100000
100000
100000
100000


/tmp/ipykernel_5448/486427149.py:3: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in chunks:


100000
69765


In [7]:
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=100_000):
    total_rows += len(chunk)

print("Total rows:", total_rows)

/tmp/ipykernel_5448/1391801591.py:3: DtypeWarning: Columns (6: store_and_fwd_flag) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(file_path, chunksize=100_000):


Total rows: 1369765


In [8]:
import pandas as pd

chunk_size = 100_000
total_rows = 0
null_counts = None
minimums = {}
maximums = {}
column_types = {}

for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):

    total_rows += len(chunk)

    # count missing values
    chunk_nulls = chunk.isnull().sum()

    if null_counts is None:
        null_counts = chunk_nulls
    else:
        null_counts += chunk_nulls

    # collect data types, minimums, and maximums
    for col in chunk.columns:
        column_types[col] = str(chunk[col].dtype)

        col_min = chunk[col].min() if chunk[col].notna().any() else None
        col_max = chunk[col].max() if chunk[col].notna().any() else None

        if col_min is not None:
            if col not in minimums or col_min < minimums[col]:
                minimums[col] = col_min

        if col_max is not None:
            if col not in maximums or col_max > maximums[col]:
                maximums[col] = col_max

print("Total rows:", total_rows)

Total rows: 1369765


In [9]:
profile = pd.DataFrame({
    "Data Type": column_types,
    "Null Count": null_counts,
    "Null %": (null_counts / total_rows * 100).round(2),
    "Minimum": minimums,
    "Maximum": maximums
})

profile

,Data Type,Null Count,Null %,Minimum,Maximum
VendorID,float64,98352,7.18,1,2
tpep_pickup_datetime,str,0,0.00,2008-12-31 23:05:14,2021-02-22 16:52:16
tpep_dropoff_datetime,str,0,0.00,2008-12-31 23:07:22,2021-02-22 16:56:15
passenger_count,float64,98352,7.18,0,8
trip_distance,float64,0,0.00,0.0,263163.28
RatecodeID,float64,98352,7.18,1,99
store_and_fwd_flag,float64,98352,7.18,N,Y
PULocationID,int64,0,0.00,1,265
DOLocationID,int64,0,0.00,1,265
payment_type,float64,98352,7.18,1,4


In [11]:
problems = {
    "Invalid pickup dates": 0,
    "Invalid drop-off dates": 0,
    "Negative or zero fares": 0,
    "Negative or zero distances": 0,
    "Zero passengers": 0,
    "Duplicate records": 0
}

start_date = pd.Timestamp("2021-01-01")
end_date = pd.Timestamp("2021-02-01")

seen_rows = set()

for chunk in pd.read_csv(file_path, chunksize=100_000, low_memory=False):

    # convert dates
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"],
        errors="coerce"
    )

    dropoff = pd.to_datetime(
        chunk["tpep_dropoff_datetime"],
        errors="coerce"
    )

    # count invalid dates
    problems["Invalid pickup dates"] += (
        (pickup < start_date) |
        (pickup >= end_date) |
        pickup.isna()
    ).sum()

    problems["Invalid drop-off dates"] += (
        (dropoff < start_date) |
        (dropoff >= end_date) |
        dropoff.isna()
    ).sum()

    # count invalid fares and distances
    problems["Negative or zero fares"] += (
        chunk["fare_amount"] <= 0
    ).sum()

    problems["Negative or zero distances"] += (
        chunk["trip_distance"] <= 0
    ).sum()

    # count zero passengers
    problems["Zero passengers"] += (
        chunk["passenger_count"] == 0
    ).sum()

    # identify duplicates across all chunks
    row_hashes = pd.util.hash_pandas_object(
        chunk,
        index=False
    )

    for row_hash in row_hashes:
        if row_hash in seen_rows:
            problems["Duplicate records"] += 1
        else:
            seen_rows.add(row_hash)

In [12]:
quality_report = pd.DataFrame(
    problems.items(),
    columns=["Problem", "Number of Records"]
)

quality_report["Percentage"] = (
    quality_report["Number of Records"] / total_rows * 100
).round(2)

quality_report

,Problem,Number of Records,Percentage
0,Invalid pickup dates,24,0.00
1,Invalid drop-off dates,137,0.01
2,Negative or zero fares,7411,0.54
3,Negative or zero distances,19952,1.46
4,Zero passengers,26726,1.95
5,Duplicate records,0,0.00


In [13]:
def clean(df):
    cleaned = df.copy()

    removed = {
        "invalid_pickup_date": 0,
        "duplicate_rows": 0
    }

    # convert date columns
    cleaned["tpep_pickup_datetime"] = pd.to_datetime(
        cleaned["tpep_pickup_datetime"],
        errors="coerce"
    )

    cleaned["tpep_dropoff_datetime"] = pd.to_datetime(
        cleaned["tpep_dropoff_datetime"],
        errors="coerce"
    )

    # remove records outside January 2021
    valid_pickup = (
        (cleaned["tpep_pickup_datetime"] >= "2021-01-01") &
        (cleaned["tpep_pickup_datetime"] < "2021-02-01")
    )

    removed["invalid_pickup_date"] = int((~valid_pickup).sum())
    cleaned = cleaned.loc[valid_pickup].copy()

    # remove duplicate rows
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    removed["duplicate_rows"] = before - len(cleaned)

    # flag suspicious records
    cleaned["flag_invalid_dropoff"] = (
        cleaned["tpep_dropoff_datetime"].isna() |
        (cleaned["tpep_dropoff_datetime"] < "2021-01-01") |
        (cleaned["tpep_dropoff_datetime"] >= "2021-02-01")
    )

    cleaned["flag_invalid_fare"] = (
        cleaned["fare_amount"] <= 0
    )

    cleaned["flag_invalid_distance"] = (
        cleaned["trip_distance"] <= 0
    )

    cleaned["flag_zero_passengers"] = (
        cleaned["passenger_count"] == 0
    )

    return cleaned, removed

In [15]:
cleaned_df, removed_rows = clean(df)

print("Original rows:", len(df))
print("Cleaned rows:", len(cleaned_df))
print("Rows removed:", removed_rows)

cleaned_df.head(10)

Original rows: 100
Cleaned rows: 100
Rows removed: {'invalid_pickup_date': 0, 'duplicate_rows': 0}


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,flag_invalid_dropoff,flag_invalid_fare,flag_invalid_distance,flag_zero_passengers
0,1,2021-01-01 00:30:10,2021-01-01 00:36:12,1,2.10,1,N,142,43,2,...,0.5,0.00,0.0,0.3,11.80,2.5,False,False,False,False
1,1,2021-01-01 00:51:20,2021-01-01 00:52:19,1,0.20,1,N,238,151,2,...,0.5,0.00,0.0,0.3,4.30,0.0,False,False,False,False
2,1,2021-01-01 00:43:30,2021-01-01 01:11:06,1,14.70,1,N,132,165,1,...,0.5,8.65,0.0,0.3,51.95,0.0,False,False,False,False
3,1,2021-01-01 00:15:48,2021-01-01 00:31:01,0,10.60,1,N,138,132,1,...,0.5,6.05,0.0,0.3,36.35,0.0,False,False,False,True
4,2,2021-01-01 00:31:49,2021-01-01 00:48:21,1,4.94,1,N,68,33,1,...,0.5,4.06,0.0,0.3,24.36,2.5,False,False,False,False
5,1,2021-01-01 00:16:29,2021-01-01 00:24:30,1,1.60,1,N,224,68,1,...,0.5,2.35,0.0,0.3,14.15,2.5,False,False,False,False
6,1,2021-01-01 00:00:28,2021-01-01 00:17:28,1,4.10,1,N,95,157,2,...,0.5,0.00,0.0,0.3,17.30,0.0,False,False,False,False
7,1,2021-01-01 00:12:29,2021-01-01 00:30:34,1,5.70,1,N,90,40,2,...,0.5,0.00,0.0,0.3,21.80,2.5,False,False,False,False
8,1,2021-01-01 00:39:16,2021-01-01 01:00:13,1,9.10,1,N,97,129,4,...,0.5,0.00,0.0,0.3,28.80,0.0,False,False,False,False
9,1,2021-01-01 00:26:12,2021-01-01 00:39:46,2,2.70,1,N,263,142,1,...,0.5,3.15,0.0,0.3,18.95,2.5,False,False,False,False


In [16]:
cleaning_results = {
    "invalid_pickup_date": 0,
    "duplicate_rows": 0
}

total_original = 0
total_cleaned = 0

for chunk in pd.read_csv(file_path, chunksize=100_000, low_memory=False):

    total_original += len(chunk)

    cleaned_chunk, removed = clean(chunk)

    total_cleaned += len(cleaned_chunk)

    for rule, count in removed.items():
        cleaning_results[rule] += count

print("Original rows:", total_original)
print("Cleaned rows:", total_cleaned)
print("Rows removed:", cleaning_results)

Original rows: 1369765
Cleaned rows: 1369741
Rows removed: {'invalid_pickup_date': 24, 'duplicate_rows': 0}


In [17]:
summary = pd.DataFrame({
    "Metric": [
        "Original Records",
        "Cleaned Records",
        "Removed Records"
    ],
    "Count": [
        total_original,
        total_cleaned,
        total_original - total_cleaned
    ]
})

summary

,Metric,Count
0,Original Records,1369765
1,Cleaned Records,1369741
2,Removed Records,24


In [19]:
print("Original columns:", len(df.columns))
print("Cleaned columns:", len(cleaned_df.columns))

Original columns: 18
Cleaned columns: 22
